# MLP Residual Alignment

How does MLP output relate to the residual stream?
Cosine alignment, parallel/perpendicular decomposition,
contribution ratio, and unembedding projection.

## Why This Matters

MLP residual alignment provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_residual_alignment import (
    mlp_residual_cosine, mlp_parallel_perpendicular,
    mlp_contribution_ratio, mlp_unembed_alignment,
    mlp_residual_alignment_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## MLP-Residual Cosine

Is the MLP reinforcing or redirecting the residual stream?

In [ ]:
for layer in range(4):
    result = mlp_residual_cosine(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_cos={result['mean_cosine']:.4f}, "
          f"{result['n_reinforcing']}/{len(result['per_position'])} reinforcing")

## Parallel/Perpendicular Decomposition

In [ ]:
for layer in range(4):
    result = mlp_parallel_perpendicular(model, tokens, layer=layer, position=-1)
    reinf = 'REINF' if result['is_reinforcing'] else 'REDIR'
    print(f"  Layer {layer}: par={result['parallel_magnitude']:.4f}, "
          f"perp={result['perpendicular_magnitude']:.4f}, "
          f"frac={result['parallel_fraction']:.3f} [{reinf}]")

## MLP Unembed Alignment

What tokens does the MLP output promote/suppress?

In [ ]:
for layer in range(4):
    result = mlp_unembed_alignment(model, tokens, layer=layer, position=-1, top_k=3)
    promoted = [(t, f'{l:.3f}') for t, l in result['promoted']]
    suppressed = [(t, f'{l:.3f}') for t, l in result['suppressed']]
    print(f"  Layer {layer}: promotes={promoted}, suppresses={suppressed}")

## Alignment Summary

In [ ]:
result = mlp_residual_alignment_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: cos={p['mean_cosine']:.4f}, "
          f"ratio={p['mean_ratio']:.4f}, reinf={p['n_reinforcing']}")